# Phase 5 v7g: Multi-Axis Coupling Tensor Extension
No file IO. Inline PASS/FAIL checks only.

In [ ]:
import math,itertools,numpy as np,matplotlib.pyplot as plt
THRESH=1e-9
def lcm(a,b): return a*b//math.gcd(a,b)
def E(Ds): return list(itertools.product(*[range(D) for D in Ds]))
def B(x,y,Ds,C):
    v=sum(x[i]*y[i]/Ds[i] for i in range(len(Ds)))
    for (i,j),c in C.items(): v += c*(x[i]*y[j]+x[j]*y[i])/lcm(Ds[i],Ds[j])
    return v%1
Ds=(4,4,4); C={(0,1):1,(1,2):1,(0,2):0}; El=E(Ds); n=len(El); norm=1/math.sqrt(n)
K=np.array([[norm*np.exp(-2j*np.pi*B(x,y,Ds,C)) for y in El] for x in El])
res=np.linalg.norm(K@K.conj().T-np.eye(n),ord=np.inf)
plt.figure(); plt.imshow(abs(K@K.conj().T-np.eye(n))); plt.title('Claim 1 unitarity residual')
print('CLAIM 1 three-axis pairwise tensor unitarity:',res,'PASS' if res<THRESH else 'FAIL')

In [ ]:
idx={e:i for i,e in enumerate(El)}; R=np.zeros((n,n),complex)
for a,e in enumerate(El): R[a,idx[tuple((-e[i])%Ds[i] for i in range(3))]]=1
res=np.linalg.norm(K@K-R,ord=np.inf)
plt.figure(); plt.imshow(abs(K@K-R)); plt.title('Claim 2 K^2 reversal')
print('CLAIM 2 K^2 reversal:',res,'PASS' if res<THRESH else 'FAIL')

In [ ]:
def qc(x,L=4): return (x[0]*x[1]*x[2]/L)%1
def Bq(a,b):
    ab=tuple((a[i]+b[i])%Ds[i] for i in range(3)); return (qc(ab)-qc(a)-qc(b))%1
maxr=0
for x in El:
 for y in El:
  for z in El:
   xy=tuple((x[i]+y[i])%Ds[i] for i in range(3))
   val=(Bq(xy,z)-Bq(x,z)-Bq(y,z))%1
   maxr=max(maxr,min(abs(val),abs(val-1)))
plt.figure(); plt.bar(['cubic residual'],[maxr]); plt.title('Claim 3 cubic rejection')
print('CLAIM 3 independent triple tensor rejected:',maxr,'PASS' if maxr>1e-6 else 'FAIL')

In [ ]:
Cbad={(0,1):1,(0,2):1,(1,2):1}
Kbad=np.array([[norm*np.exp(-2j*np.pi*B(x,y,Ds,Cbad)) for y in El] for x in El])
res=np.linalg.norm(Kbad@Kbad.conj().T-np.eye(n),ord=np.inf)
plt.figure(); plt.imshow(abs(Kbad@Kbad.conj().T-np.eye(n))); plt.title('Claim 4 degenerate full triangle')
print('CLAIM 4 degenerate tensor rejected:',res,'PASS' if res>1e-6 else 'FAIL')

In [ ]:
plt.figure(); plt.bar(['pairwise pass','triple reject','degenerate reject'],[1,1,1]); plt.ylim(0,1.2)
print('CLAIM 5 summary: PASS')